In [ ]:
"""
=============================================================================
 ÉTUDES 5 ET 6 — COMPARAISON END-TO-END ET VALIDATION DES PONDÉRATIONS
=============================================================================
 PFE — Trustworthy AI for Opportunistic Screening of Cardiovascular Risk
 Abdallah OUAHMANE — FSR Rabat

 ÉTUDE 5 — Le pipeline modulaire bat-il un classificateur unique ?
   Compare la formule V5 au Module 4 employé seul, sans aucune fusion, et
   quantifie l'information que les composantes anatomiques ajoutent à ce
   dernier.

 ÉTUDE 6 — Les pondérations de V5 résistent-elles à la validation ?
   Les quatre poids ont été fixés par optimisation sur les 501 images qui
   servent aussi à l'évaluation. Une validation par demi-échantillons
   répétés mesure la part de gain imputable au surajustement, et une
   analyse de stabilité examine la dispersion des poids optimaux.

 ENTRÉE : resultats_phase4_v5.csv
=============================================================================
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold

plt.rcParams['font.family'] = 'DejaVu Sans'

CSV_PATH = '/kaggle/input/datasets/abdallahouahmane/resultas-phase4/resultats_phase4_v5.csv'

# Pondérations de la formule déployée
W = {'ctr': 0.15, 'cong': 0.40, 'calcif': 0.10, 'cardio': 0.35}

# Planchers d'auditabilité imposés lors de l'optimisation
PLANCHER = {'ctr': 0.15, 'cong': 0.15, 'calcif': 0.10}
PLAFOND_CARDIO = 0.40

N_SPLITS = 200
N_BOOT = 2000
SEED = 42
PAS = 0.05

rng = np.random.default_rng(SEED)


# ---------------------------------------------------------------------------
# Test de DeLong (Sun & Xu, 2014)
# ---------------------------------------------------------------------------
def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N); T2[J] = T
    return T2


def delong(y, a, b):
    """Retourne (delta, erreur standard, p-value)."""
    o = np.argsort(-y); yy = y[o]
    m = int(yy.sum()); n = len(yy) - m
    P = np.vstack([a[o], b[o]])
    tx = np.empty([2, m]); ty = np.empty([2, n]); tz = np.empty([2, m + n])
    for r in range(2):
        tx[r] = _midrank(P[r, :m])
        ty[r] = _midrank(P[r, m:])
        tz[r] = _midrank(P[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    cov = np.atleast_2d(np.cov((tz[:, :m] - tx) / n) / m
                        + np.cov(1.0 - (tz[:, m:] - ty) / m) / n)
    d = aucs[0] - aucs[1]
    se = np.sqrt(max(cov[0, 0] + cov[1, 1] - 2 * cov[0, 1], 1e-30))
    return d, se, 2 * (1 - stats.norm.cdf(abs(d / se)))


def auc_ic(y, s, n_boot=N_BOOT):
    """AUC observée et intervalle de confiance à 95 % par bootstrap stratifié."""
    ip, iN = np.where(y == 1)[0], np.where(y == 0)[0]
    b = []
    for _ in range(n_boot):
        idx = np.r_[rng.choice(ip, len(ip), True), rng.choice(iN, len(iN), True)]
        b.append(roc_auc_score(y[idx], s[idx]))
    return roc_auc_score(y, s), np.percentile(b, 2.5), np.percentile(b, 97.5)


# ---------------------------------------------------------------------------
# Chargement
# ---------------------------------------------------------------------------
df = pd.read_csv(CSV_PATH)
y = df['label_reel'].apply(
    lambda x: 1 if ('Cardiomegaly' in str(x) or 'Edema' in str(x)) else 0
).values.astype(int)

s_ctr    = df['s_ctr'].values
s_cal    = df['s_calcif'].values
s_cong   = df['s_cong'].values
s_cardio = df['prob_cardio'].values
s_m4     = 1.0 - df['prob_normal'].values      # signal complet du Module 4

COMPOSANTES = {'ctr': s_ctr, 'cong': s_cong,
               'calcif': s_cal, 'cardio': s_cardio}


def fusion(p):
    return sum(p[c] * COMPOSANTES[c] for c in COMPOSANTES)


v5 = fusion(W)
auc_v5 = roc_auc_score(y, v5)

# Formules antérieures, conservées comme repères
v1 = 0.50 * s_ctr + 0.25 * s_cal + 0.25 * s_cong
v3 = 0.45 * s_ctr + 0.55 * s_cong
v4 = 0.35 * s_ctr + 0.45 * s_cong + 0.20 * s_cal

print("=" * 70)
print("  ÉTUDES 5 ET 6")
print("=" * 70)
print(f"\n  {len(df)} images | {y.sum()} malades | {(y == 0).sum()} sains")
print(f"  Formule V5 : " + " + ".join(f"{W[c]}×{c}" for c in W))
print(f"  AUC de référence : {auc_v5:.4f}")


# ===========================================================================
# ÉTUDE 5 — COMPARAISON AU CLASSIFICATEUR END-TO-END
# ===========================================================================
print("\n" + "=" * 70)
print("  ÉTUDE 5 — LE PIPELINE BAT-IL UN CLASSIFICATEUR UNIQUE ?")
print("=" * 70)
print("\n  Le Module 4 produit trois probabilités. Le comparateur")
print("  1 - prob[Normal] agrège les deux pathologies et se calcule sans")
print("  aucune fusion. La formule V5 en exploite deux composantes")
print("  (prob[Edema] et prob[Cardiomegaly]) auxquelles elle ajoute deux")
print("  signaux anatomiques.\n")

print(f"  {'Configuration':<38}{'AUC':>9}{'IC 95 %':>21}")
print("  " + "-" * 66)
for nom, sc in [('Fusion uniforme (0.50/0.25/0.25)', v1),
                ('Pipeline V3 (0.45/0.55)', v3),
                ('Optimum V4 (0.35/0.45/0.20)', v4),
                ('Pipeline V5 (déployé)', v5),
                ('Module 4 seul (1 - prob[Normal])', s_m4)]:
    a, lo, hi = auc_ic(y, sc)
    print(f"  {nom:<38}{a:>9.4f}   [{lo:.4f} – {hi:.4f}]")

print(f"\n  Tests de DeLong, V5 contre chaque référence :")
print("  " + "-" * 66)
for nom, ref in [('V5 vs fusion unif.', v1), ('V5 vs V3', v3),
                 ('V5 vs V4', v4), ('V5 vs M4 seul', s_m4)]:
    d, se, p = delong(y, v5, ref)
    verdict = "significatif" if p < 0.05 else "non significatif"
    print(f"    {nom:<22} ΔAUC = {d:+.4f} ± {se:.4f}   "
          f"p = {p:.4f}   {verdict}")

# --- Part du Module 4 dans la formule V5 -------------------------------
print(f"\n  Part respective des sources dans la formule V5 :")
print("  " + "-" * 66)
part_m4 = W['cong'] + W['cardio']
part_anat = W['ctr'] + W['calcif']
print(f"    Composantes issues du Module 4 : {part_m4:.2f}")
print(f"    Composantes anatomiques        : {part_anat:.2f}")

sans_anat = (W['cong'] * s_cong + W['cardio'] * s_cardio) / part_m4
a_sans, _, _ = auc_ic(y, sans_anat, 500)
print(f"\n    V5 réduite aux composantes M4 : AUC = {a_sans:.4f}")
print(f"    V5 complète                   : AUC = {auc_v5:.4f}")
print(f"    Apport des composantes anatomiques : {auc_v5 - a_sans:+.4f}")

# --- Le CTR ajoute-t-il au signal complet de M4 ? ----------------------
print(f"\n  Le CTR ajoute-t-il de l'information au Module 4 complet ?")
print("  " + "-" * 66)
grille = np.arange(0, 1.0001, PAS)
best_m4 = max(((w, roc_auc_score(y, w * s_ctr + (1 - w) * s_m4))
               for w in grille), key=lambda t: t[1])
print(f"    Optimum de w × s_ctr + (1-w) × s_m4 : "
      f"w = {best_m4[0]:.2f}, AUC = {best_m4[1]:.4f}")

r, pv = stats.pearsonr(s_ctr, s_m4)
print(f"    Corrélation s_ctr / s_m4 : r = {r:+.3f} (p = {pv:.4f})")

cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
p1 = cross_val_predict(LogisticRegression(max_iter=1000), np.c_[s_m4], y,
                       cv=cv, method='predict_proba')[:, 1]
p2 = cross_val_predict(LogisticRegression(max_iter=1000),
                       np.c_[s_m4, s_ctr, s_cal], y,
                       cv=cv, method='predict_proba')[:, 1]
print(f"    Régression logistique — M4 seul        : {roc_auc_score(y, p1):.4f}")
print(f"    Régression logistique — M4 + CTR + M3  : {roc_auc_score(y, p2):.4f}")
print(f"    Apport des signaux anatomiques : "
      f"{roc_auc_score(y, p2) - roc_auc_score(y, p1):+.4f}")

print("\n  Lecture — le critère d'évaluation est constitué des étiquettes NIH")
print("  extraites par traitement du langage naturel, que le Module 4 a été")
print("  entraîné à reproduire sur ces mêmes classes. L'AUC mesure donc la")
print("  concordance avec une annotation d'imagerie, non la valeur prédictive")
print("  sur la survenue d'événements cardiovasculaires. Le Module 4 est")
print("  structurellement avantagé par ce protocole.")


# ===========================================================================
# ÉTUDE 6 — VALIDATION DES PONDÉRATIONS
# ===========================================================================
print("\n" + "=" * 70)
print("  ÉTUDE 6 — LES PONDÉRATIONS RÉSISTENT-ELLES À LA VALIDATION ?")
print("=" * 70)

print(f"\n  AUC des composantes prises isolément :")
print(f"\n  {'Composante':<28}{'AUC':>9}{'IC 95 %':>21}")
print("  " + "-" * 56)
for nom, sc in [('s_ctr (Module 1)', s_ctr),
                ('s_calcif (Module 3)', s_cal),
                ('s_cong (Module 4, œdème)', s_cong),
                ('prob_cardio (Module 4)', s_cardio)]:
    a, lo, hi = auc_ic(y, sc)
    print(f"  {nom:<28}{a:>9.4f}   [{lo:.4f} – {hi:.4f}]")


def grille_contrainte():
    """Configurations admissibles sous contrainte d'auditabilité."""
    out = []
    for a in np.arange(PLANCHER['ctr'], 0.51, PAS):
        for c in np.arange(PLANCHER['calcif'], 0.41, PAS):
            for d in np.arange(0.0, PLAFOND_CARDIO + 1e-9, PAS):
                b = 1 - a - c - d
                if b < PLANCHER['cong']:
                    continue
                out.append({'ctr': round(a, 2), 'cong': round(b, 2),
                            'calcif': round(c, 2), 'cardio': round(d, 2)})
    return out


CONFIGS = grille_contrainte()
print(f"\n  Recherche exhaustive sous contrainte "
      f"({len(CONFIGS)} configurations, pas de {PAS}) :")
print(f"    w_ctr ≥ {PLANCHER['ctr']}, w_cong ≥ {PLANCHER['cong']}, "
      f"w_calcif ≥ {PLANCHER['calcif']}, w_cardio ≤ {PLAFOND_CARDIO}")
print("  " + "-" * 56)

res = sorted(((roc_auc_score(y, fusion(p)), p) for p in CONFIGS),
             key=lambda t: -t[0])
print(f"  {'AUC':>9}   {'w_ctr':>6}{'w_cong':>8}{'w_calcif':>10}{'w_cardio':>10}")
for a, p in res[:6]:
    print(f"  {a:>9.4f}   {p['ctr']:>6.2f}{p['cong']:>8.2f}"
          f"{p['calcif']:>10.2f}{p['cardio']:>10.2f}")

auc_opt = res[0][0]
rang_v5 = sum(1 for a, _ in res if a > auc_v5) + 1
print(f"\n  Optimum de la grille : {auc_opt:.4f}")
print(f"  Formule V5 déployée  : {auc_v5:.4f}  ({auc_v5 - auc_opt:+.4f})")
print(f"  Rang de V5 : {rang_v5} sur {len(res)}")

# --- Validation par demi-échantillons répétés --------------------------
print(f"\n  Validation par {N_SPLITS} partitions aléatoires stratifiées :")
print("  " + "-" * 56)
print("  Les poids sont optimisés sur une moitié puis évalués sur l'autre.")
print("  L'écart entre les deux mesure la part de gain non reproductible.")

ip, iN = np.where(y == 1)[0], np.where(y == 0)[0]
poids, auc_opt_b, auc_v5_b = [], [], []

for _ in range(N_SPLITS):
    pp, nn = rng.permutation(ip), rng.permutation(iN)
    A = np.r_[pp[:len(pp) // 2], nn[:len(nn) // 2]]
    B = np.r_[pp[len(pp) // 2:], nn[len(nn) // 2:]]

    meilleur, a_max = None, -1.0
    for p in CONFIGS:
        a = roc_auc_score(y[A], fusion(p)[A])
        if a > a_max:
            meilleur, a_max = p, a

    poids.append([meilleur[c] for c in ['ctr', 'cong', 'calcif', 'cardio']])
    auc_opt_b.append(roc_auc_score(y[B], fusion(meilleur)[B]))
    auc_v5_b.append(roc_auc_score(y[B], v5[B]))

P = np.array(poids)
auc_opt_b, auc_v5_b = np.array(auc_opt_b), np.array(auc_v5_b)

print(f"\n  {'Poids':<12}{'V5':>7}{'médiane':>10}{'IQR':>18}")
print("  " + "-" * 48)
for i, nom in enumerate(['ctr', 'cong', 'calcif', 'cardio']):
    print(f"  w_{nom:<10}{W[nom]:>7.2f}{np.median(P[:, i]):>10.2f}"
          f"   [{np.percentile(P[:, i], 25):.2f} – "
          f"{np.percentile(P[:, i], 75):.2f}]")

print(f"\n  AUC sur la moitié d'évaluation, jamais vue par l'optimisation :")
print(f"    poids réoptimisés : {auc_opt_b.mean():.4f} ± {auc_opt_b.std():.4f}")
print(f"    poids V5 fixes    : {auc_v5_b.mean():.4f} ± {auc_v5_b.std():.4f}")
print(f"    écart moyen       : {(auc_opt_b - auc_v5_b).mean():+.4f}")

surajust = auc_v5 - auc_v5_b.mean()
print(f"\n  Estimation du surajustement :")
print(f"    AUC de V5 sur l'échantillon complet   : {auc_v5:.4f}")
print(f"    AUC de V5 en évaluation indépendante  : {auc_v5_b.mean():.4f}")
print(f"    écart : {surajust:+.4f}")

# --- Contribution de M3 sous validation --------------------------------
print(f"\n  Le Module 3 contribue-t-il sous validation indépendante ?")
print("  " + "-" * 56)
somme_sans = W['ctr'] + W['cong'] + W['cardio']
sans_m3 = {'ctr': W['ctr'] / somme_sans, 'cong': W['cong'] / somme_sans,
           'calcif': 0.0, 'cardio': W['cardio'] / somme_sans}
v5_sans_m3 = fusion(sans_m3)

ecarts = []
for _ in range(N_SPLITS):
    pp, nn = rng.permutation(ip), rng.permutation(iN)
    B = np.r_[pp[len(pp) // 2:], nn[len(nn) // 2:]]
    ecarts.append(roc_auc_score(y[B], v5[B])
                  - roc_auc_score(y[B], v5_sans_m3[B]))
ecarts = np.array(ecarts)

d, se, p_m3 = delong(y, v5, v5_sans_m3)
print(f"    V5 avec M3    : {auc_v5:.4f}")
print(f"    V5 sans M3    : {roc_auc_score(y, v5_sans_m3):.4f}")
print(f"    apport        : {auc_v5 - roc_auc_score(y, v5_sans_m3):+.4f}")
print(f"    DeLong        : p = {p_m3:.4f} → "
      f"{'significatif' if p_m3 < 0.05 else 'NON significatif'}")
print(f"    apport médian sur {N_SPLITS} partitions : {np.median(ecarts):+.4f}")
print(f"    positif dans {np.mean(ecarts > 0) * 100:.1f} % des partitions")

# --- Verdict -----------------------------------------------------------
print("\n" + "=" * 70)
print("  VERDICT — ÉTUDE 6")
print("=" * 70)
print(f"\n  Les pondérations de V5 coïncident avec l'optimum de la grille")
print(f"  contrainte (écart {auc_v5 - auc_opt:+.4f}) et se situent au rang")
print(f"  {rang_v5} sur {len(res)} configurations admissibles.")
print(f"\n  Sous validation par demi-échantillons, les poids fixes de V5")
print(f"  atteignent {auc_v5_b.mean():.4f} contre {auc_opt_b.mean():.4f} pour des poids")
print(f"  réoptimisés à chaque partition, soit un écart de "
      f"{(auc_opt_b - auc_v5_b).mean():+.4f}.")
print(f"  Le surajustement de la formule déployée est estimé à "
      f"{surajust:+.4f} d'AUC.")
print(f"\n  L'AUC honnête à rapporter est celle de l'évaluation")
print(f"  indépendante : {auc_v5_b.mean():.4f} ± {auc_v5_b.std():.4f}.")

# ===========================================================================
# FIGURE
# ===========================================================================
fig, axes = plt.subplots(1, 4, figsize=(23, 5.2))

ax = axes[0]
for nom, sc, col in [('Fusion uniforme', v1, '#95A5A6'),
                     ('Pipeline V3', v3, '#7F8C8D'),
                     ('Pipeline V5', v5, '#C0392B'),
                     ('Module 4 seul', s_m4, '#1E8449')]:
    f, t, _ = roc_curve(y, sc)
    ax.plot(f, t, lw=2, color=col, label=f"{nom} — {roc_auc_score(y, sc):.3f}")
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('1 − Spécificité'); ax.set_ylabel('Sensibilité')
ax.set_title('Étude 5 — pipeline et classificateur unique', fontweight='bold')
ax.legend(loc='lower right', fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
largeur = 0.19
x = np.arange(4)
for i, nom in enumerate(['ctr', 'cong', 'calcif', 'cardio']):
    ax.boxplot([P[:, i]], positions=[i], widths=0.55, patch_artist=True,
               boxprops=dict(facecolor='#DCE6F1'), showfliers=False)
    ax.scatter([i], [W[nom]], color='#C0392B', s=90, zorder=5,
               marker='D', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(['w_ctr', 'w_cong', 'w_calcif', 'w_cardio'], fontsize=9)
ax.set_ylabel('Poids optimal par partition')
ax.set_title(f'Étude 6 — stabilité des poids\n{N_SPLITS} partitions '
             '(losange : V5)', fontweight='bold')
ax.grid(alpha=0.3, axis='y')

ax = axes[2]
ax.boxplot([auc_v5_b, auc_opt_b],
           tick_labels=['V5 (poids fixes)', 'Poids réoptimisés'],
           patch_artist=True, notch=True,
           boxprops=dict(facecolor='#DCE6F1'))
ax.axhline(auc_v5, color='#C0392B', ls='--', lw=1.8,
           label=f'V5 sur échantillon complet ({auc_v5:.4f})')
ax.set_ylabel("AUC sur la moitié d'évaluation")
ax.set_title('Étude 6 — validation indépendante', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='y')

ax = axes[3]
ax.hist(ecarts, bins=30, color='#7D3C98', edgecolor='white', alpha=0.8)
ax.axvline(0, color='black', lw=1.4)
ax.axvline(np.median(ecarts), color='#C0392B', ls='--', lw=1.8,
           label=f'médiane ({np.median(ecarts):+.4f})')
ax.set_xlabel("Apport de M3 à l'AUC")
ax.set_ylabel('Effectif')
ax.set_title(f'Étude 6 — apport du Module 3\nsur {N_SPLITS} partitions',
             fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
for ext in ('png', 'svg'):
    plt.savefig(f'etudes_5_6_v5.{ext}', dpi=300, bbox_inches='tight',
                facecolor='white')
print("\n  Figure : etudes_5_6_v5.png")
print("  Terminé.")